In [4]:
# ============================================================
# URBAN MOBILITY INTELLIGENCE SYSTEM
# Notebook 06 — Excel Executive Report
# Professional multi-sheet Excel workbook
# ============================================================

import pandas as pd
import numpy as np
import os
from openpyxl import Workbook
from openpyxl.styles import (
    PatternFill, Font, Alignment, Border, Side, GradientFill
)
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.chart.series import DataPoint
import warnings
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────
RAW_DATA_PATH       = r"D:\Projects\End-to-end projects\14. Urban Mobility Intelligence\data\raw"
PROCESSED_DATA_PATH = r"D:\Projects\End-to-end projects\14. Urban Mobility Intelligence\data\processed"
EXCEL_OUTPUT_PATH   = r"D:\Projects\End-to-end projects\14. Urban Mobility Intelligence\outputs\excel"

# ── Load all data ────────────────────────────────────────────
print("Loading data...")

rides_df    = pd.read_csv(os.path.join(RAW_DATA_PATH, "rides.csv"))
zones_df    = pd.read_csv(os.path.join(RAW_DATA_PATH, "zones.csv"))
drivers_df  = pd.read_csv(os.path.join(RAW_DATA_PATH, "drivers.csv"))
stations_df = pd.read_csv(os.path.join(RAW_DATA_PATH, "charging_stations.csv"))
events_df   = pd.read_csv(os.path.join(RAW_DATA_PATH, "events.csv"))
zone_map    = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, "zone_map_data.csv"))
forecast_df = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, "citywide_demand_forecast.csv"))

rides_df['date']    = pd.to_datetime(rides_df['date'])
forecast_df['date'] = pd.to_datetime(forecast_df['date'])

print(f"  ✓ All datasets loaded")

# ── Color palette for Excel ──────────────────────────────────
DARK_BLUE   = "1a1a2e"
MID_BLUE    = "16213e"
ACCENT_BLUE = "0f3460"
RAPIDO_RED  = "e94560"
WHITE       = "FFFFFF"
LIGHT_GREY  = "F5F5F5"
DARK_GREY   = "333333"
GREEN       = "2e7d32"
ORANGE      = "f57c00"
LIGHT_BLUE  = "e3f2fd"
HEADER_BLUE = "1565c0"

def make_fill(hex_color):
    return PatternFill(
        start_color=hex_color,
        end_color=hex_color,
        fill_type="solid"
    )

def make_font(bold=False, color=WHITE, size=11, italic=False):
    return Font(
        bold=bold, color=color,
        size=size, italic=italic,
        name='Calibri'
    )

def make_border():
    side = Side(style='thin', color='CCCCCC')
    return Border(
        left=side, right=side,
        top=side,  bottom=side
    )

def make_center():
    return Alignment(
        horizontal='center',
        vertical='center',
        wrap_text=True
    )

def make_left():
    return Alignment(
        horizontal='left',
        vertical='center',
        wrap_text=True
    )

print("  ✓ Style helpers ready")
print("\n✓ Block 1 complete - Excel environment ready.")

Loading data...
  ✓ All datasets loaded
  ✓ Style helpers ready

✓ Block 1 complete - Excel environment ready.


In [5]:
# ============================================================
# BLOCK 2 — Build Full Executive Excel Workbook
# 6 sheets: Summary, Zone Intelligence, Driver Equity,
#           Demand Forecast, EV Charging, ML Model Results
# ============================================================

wb = Workbook()
wb.remove(wb.active)  # remove default empty sheet

# ============================================================
# SHEET 1 — EXECUTIVE SUMMARY
# ============================================================
ws1 = wb.create_sheet("Executive Summary")
ws1.sheet_view.showGridLines = False
ws1.column_dimensions['A'].width = 35
ws1.column_dimensions['B'].width = 22
ws1.column_dimensions['C'].width = 22
ws1.column_dimensions['D'].width = 22
ws1.column_dimensions['E'].width = 22

# Title block
ws1.merge_cells('A1:E1')
ws1['A1'] = '🏙️ URBAN MOBILITY INTELLIGENCE SYSTEM'
ws1['A1'].font      = make_font(bold=True, size=18)
ws1['A1'].fill      = make_fill(DARK_BLUE)
ws1['A1'].alignment = make_center()
ws1.row_dimensions[1].height = 40

ws1.merge_cells('A2:E2')
ws1['A2'] = 'Bangalore Fleet Intelligence Report | Rapido | 2024'
ws1['A2'].font      = make_font(size=12, italic=True, color='90CAF9')
ws1['A2'].fill      = make_fill(DARK_BLUE)
ws1['A2'].alignment = make_center()
ws1.row_dimensions[2].height = 25

ws1.merge_cells('A3:E3')
ws1['A3'] = ''
ws1['A3'].fill = make_fill(DARK_BLUE)

# ── KPI Cards Row ────────────────────────────────────────────
completed   = rides_df[rides_df['status'] == 'completed']
total_rev   = completed['fare_amount'].sum()
comp_rate   = len(completed) / len(rides_df) * 100
avg_fare    = completed['fare_amount'].mean()
unmet_rate  = (rides_df['status'] == 'no_driver_found').mean() * 100
avg_wait    = rides_df['wait_time_min'].mean()
ev_share    = rides_df['vehicle_type'].str.contains('EV').mean() * 100
surge_rate  = (rides_df['surge_multiplier'] > 1.0).mean() * 100

kpis = [
    ("Total Rides",         f"{len(rides_df):,}",       ACCENT_BLUE),
    ("Total Revenue",       f"₹{total_rev/1e7:.2f} Cr", GREEN),
    ("Completion Rate",     f"{comp_rate:.1f}%",         HEADER_BLUE),
    ("Avg Fare",            f"₹{avg_fare:.0f}",          ACCENT_BLUE),
    ("Unmet Demand Rate",   f"{unmet_rate:.1f}%",        "c62828"),
]

row = 4
for col_idx, (label, value, color) in enumerate(kpis, 1):
    col = get_column_letter(col_idx)
    ws1.merge_cells(f'{col}{row}:{col}{row}')
    ws1[f'{col}{row}'] = label
    ws1[f'{col}{row}'].font      = make_font(bold=True, size=10, color='90CAF9')
    ws1[f'{col}{row}'].fill      = make_fill(color)
    ws1[f'{col}{row}'].alignment = make_center()
    ws1.row_dimensions[row].height = 20

    ws1[f'{col}{row+1}'] = value
    ws1[f'{col}{row+1}'].font      = make_font(bold=True, size=15)
    ws1[f'{col}{row+1}'].fill      = make_fill(color)
    ws1[f'{col}{row+1}'].alignment = make_center()
    ws1.row_dimensions[row+1].height = 32

kpis2 = [
    ("Active Drivers",      f"{(drivers_df['status']=='active').sum():,}", ACCENT_BLUE),
    ("EV Ride Share",       f"{ev_share:.1f}%",                            GREEN),
    ("Surge Ride Rate",     f"{surge_rate:.1f}%",                          "f57c00"),
    ("Avg Wait Time",       f"{avg_wait:.1f} mins",                        HEADER_BLUE),
    ("Zones Monitored",     "200",                                          ACCENT_BLUE),
]

row = 7
for col_idx, (label, value, color) in enumerate(kpis2, 1):
    col = get_column_letter(col_idx)
    ws1[f'{col}{row}'] = label
    ws1[f'{col}{row}'].font      = make_font(bold=True, size=10, color='90CAF9')
    ws1[f'{col}{row}'].fill      = make_fill(color)
    ws1[f'{col}{row}'].alignment = make_center()
    ws1.row_dimensions[row].height = 20

    ws1[f'{col}{row+1}'] = value
    ws1[f'{col}{row+1}'].font      = make_font(bold=True, size=15)
    ws1[f'{col}{row+1}'].fill      = make_fill(color)
    ws1[f'{col}{row+1}'].alignment = make_center()
    ws1.row_dimensions[row+1].height = 32

# ── Section: Zone Classification ─────────────────────────────
row = 10
ws1.merge_cells(f'A{row}:E{row}')
ws1[f'A{row}'] = 'ZONE INTELLIGENCE OVERVIEW'
ws1[f'A{row}'].font      = make_font(bold=True, size=12)
ws1[f'A{row}'].fill      = make_fill(HEADER_BLUE)
ws1[f'A{row}'].alignment = make_left()
ws1.row_dimensions[row].height = 22

zone_class_counts = zone_map['zone_classification'].value_counts()
zone_colors_excel = {
    'CRITICAL DEAD ZONE'   : 'c62828',
    'HIGH IDLE ZONE'       : 'e65100',
    'CRITICAL DEMAND GAP'  : '0d47a1',
    'HIGH DEMAND GAP'      : '1565c0',
    'HIGH EFFICIENCY ZONE' : '1b5e20',
    'BALANCED ZONE'        : '2e7d32',
    'NO DRIVER DATA'       : '757575',
}

headers = ['Zone Classification', 'Count', 'Share %', 'Action Required', 'Priority']
row += 1
for col_idx, h in enumerate(headers, 1):
    col = get_column_letter(col_idx)
    ws1[f'{col}{row}'] = h
    ws1[f'{col}{row}'].font      = make_font(bold=True, size=10)
    ws1[f'{col}{row}'].fill      = make_fill(DARK_GREY)
    ws1[f'{col}{row}'].alignment = make_center()
    ws1[f'{col}{row}'].border    = make_border()
    ws1.row_dimensions[row].height = 18

action_map = {
    'CRITICAL DEAD ZONE'   : ('Reposition drivers immediately', '🔴 URGENT'),
    'HIGH IDLE ZONE'       : ('Reduce driver allocation',       '🟠 HIGH'),
    'CRITICAL DEMAND GAP'  : ('Send drivers immediately',       '🔴 URGENT'),
    'HIGH DEMAND GAP'      : ('Increase driver allocation',     '🟠 HIGH'),
    'HIGH EFFICIENCY ZONE' : ('Maintain current allocation',    '🟢 GOOD'),
    'BALANCED ZONE'        : ('Monitor regularly',              '🟡 NORMAL'),
    'NO DRIVER DATA'       : ('Investigate zone',               '⚪ LOW'),
}

for cls, count in zone_class_counts.items():
    row += 1
    pct    = count / len(zone_map) * 100
    action, priority = action_map.get(cls, ('Monitor', 'NORMAL'))
    color  = zone_colors_excel.get(cls, '757575')

    data = [cls, count, f"{pct:.1f}%", action, priority]
    for col_idx, val in enumerate(data, 1):
        col = get_column_letter(col_idx)
        ws1[f'{col}{row}'] = val
        ws1[f'{col}{row}'].font      = make_font(
            bold=(col_idx==1), size=10,
            color=WHITE if col_idx==1 else DARK_GREY
        )
        ws1[f'{col}{row}'].fill = (
            make_fill(color) if col_idx==1 else make_fill(LIGHT_GREY)
        )
        ws1[f'{col}{row}'].alignment = make_center()
        ws1[f'{col}{row}'].border    = make_border()
    ws1.row_dimensions[row].height = 18

# ── Section: Key Findings ─────────────────────────────────────
row += 2
ws1.merge_cells(f'A{row}:E{row}')
ws1[f'A{row}'] = 'KEY FINDINGS & RECOMMENDATIONS'
ws1[f'A{row}'].font      = make_font(bold=True, size=12)
ws1[f'A{row}'].fill      = make_fill(HEADER_BLUE)
ws1[f'A{row}'].alignment = make_left()
ws1.row_dimensions[row].height = 22

findings = [
    ("🔴 Dead Zone Alert",
     f"20 Critical Dead Zones identified - Jayanagar 4B, JP Nagar Phase 2, Sadashivanagar have 11-13 idle drivers during peak hours"),
    ("🔵 Demand Gap Alert",
     f"6 Critical Demand Gap zones - Old Airport Road at 779 rides/driver. Immediate repositioning required."),
    ("💰 Revenue Opportunity",
     f"₹32,991 daily revenue gain possible by repositioning 39 drivers from surplus to deficit zones"),
    ("⚡ ML Prediction",
     f"XGBoost model achieves 90.5% accuracy (ROC-AUC: 0.96) with ZERO false alarms in spike prediction"),
    ("📈 Demand Forecast",
     f"94.1% accurate 60-day forecast. Peak predicted Dec 29 (Sunburn Festival) - pre-position fleet by Dec 28"),
    ("🌧️ Weather Impact",
     f"Heavy rain increases demand by up to 2.1x. Deploy surge driver protocol when rainfall > 15mm"),
    ("⚡ EV Network",
     f"₹7.33 Cr monthly charging revenue. ChargeZone leads utilization at 63%. Rapido Internal lowest at 45%."),
    ("🚗 Driver Equity",
     f"Wilson Garden drivers earn 37% below city average. 21 Critical Underpaid Zones — retention risk."),
]

for finding_title, finding_text in findings:
    row += 1
    ws1.merge_cells(f'A{row}:A{row}')
    ws1[f'A{row}'] = finding_title
    ws1[f'A{row}'].font      = make_font(bold=True, size=10, color=WHITE)
    ws1[f'A{row}'].fill      = make_fill(ACCENT_BLUE)
    ws1[f'A{row}'].alignment = make_left()
    ws1[f'A{row}'].border    = make_border()

    ws1.merge_cells(f'B{row}:E{row}')
    ws1[f'B{row}'] = finding_text
    ws1[f'B{row}'].font      = make_font(size=9, color=DARK_GREY)
    ws1[f'B{row}'].fill      = make_fill(LIGHT_GREY)
    ws1[f'B{row}'].alignment = make_left()
    ws1[f'B{row}'].border    = make_border()
    ws1.row_dimensions[row].height = 28

print("  ✓ Sheet 1: Executive Summary")

# ============================================================
# SHEET 2 — ZONE INTELLIGENCE
# ============================================================
ws2 = wb.create_sheet("Zone Intelligence")
ws2.sheet_view.showGridLines = False

ws2.merge_cells('A1:N1')
ws2['A1'] = 'ZONE INTELLIGENCE - DEAD ZONE & DEMAND GAP ANALYSIS'
ws2['A1'].font      = make_font(bold=True, size=14)
ws2['A1'].fill      = make_fill(DARK_BLUE)
ws2['A1'].alignment = make_center()
ws2.row_dimensions[1].height = 32

zone_report = zone_map[[
    'zone_id', 'zone_name', 'zone_type',
    'zone_classification', 'total_rides',
    'completed_rides', 'unmet_rides',
    'completion_rate', 'unmet_rate',
    'avg_wait_time', 'avg_fare', 'total_revenue',
    'drivers_in_zone', 'rides_per_driver',
]].copy()

zone_report.columns = [
    'Zone ID', 'Zone Name', 'Zone Type',
    'Classification', 'Total Rides',
    'Completed', 'Unmet',
    'Completion %', 'Unmet %',
    'Avg Wait (min)', 'Avg Fare (₹)', 'Revenue (₹)',
    'Drivers', 'Rides/Driver',
]

col_widths = [10,28,16,24,12,12,10,13,10,14,13,15,10,13]
for i, w in enumerate(col_widths, 1):
    ws2.column_dimensions[get_column_letter(i)].width = w

# Header row
for col_idx, col_name in enumerate(zone_report.columns, 1):
    col = get_column_letter(col_idx)
    ws2[f'{col}2'] = col_name
    ws2[f'{col}2'].font      = make_font(bold=True, size=10)
    ws2[f'{col}2'].fill      = make_fill(HEADER_BLUE)
    ws2[f'{col}2'].alignment = make_center()
    ws2[f'{col}2'].border    = make_border()
ws2.row_dimensions[2].height = 20

# Data rows
zone_report_sorted = zone_report.sort_values(
    'Classification'
).reset_index(drop=True)

for r_idx, row_data in zone_report_sorted.iterrows():
    excel_row = r_idx + 3
    cls       = row_data['Classification']
    row_color = {
        'CRITICAL DEAD ZONE'   : 'ffebee',
        'HIGH IDLE ZONE'       : 'fff3e0',
        'CRITICAL DEMAND GAP'  : 'e3f2fd',
        'HIGH DEMAND GAP'      : 'e8f4fd',
        'HIGH EFFICIENCY ZONE' : 'e8f5e9',
        'BALANCED ZONE'        : 'f9fbe7',
    }.get(cls, LIGHT_GREY)

    for col_idx, val in enumerate(row_data, 1):
        col = get_column_letter(col_idx)
        # Format numbers
        if isinstance(val, float):
            val = round(val, 1)
        ws2[f'{col}{excel_row}'] = val
        ws2[f'{col}{excel_row}'].font      = make_font(
            size=9, color=DARK_GREY,
            bold=(col_idx == 4)
        )
        ws2[f'{col}{excel_row}'].fill      = (
            make_fill(zone_colors_excel.get(cls, '757575'))
            if col_idx == 4
            else make_fill(row_color)
        )
        ws2[f'{col}{excel_row}'].alignment = make_center()
        ws2[f'{col}{excel_row}'].border    = make_border()
    ws2.row_dimensions[excel_row].height = 16

ws2.freeze_panes = 'A3'
ws2.auto_filter.ref = f"A2:{get_column_letter(len(zone_report.columns))}2"
print("  ✓ Sheet 2: Zone Intelligence")

# ============================================================
# SHEET 3 — DRIVER EARNINGS EQUITY
# ============================================================
ws3 = wb.create_sheet("Driver Earnings Equity")
ws3.sheet_view.showGridLines = False

ws3.merge_cells('A1:H1')
ws3['A1'] = 'DRIVER EARNINGS EQUITY ANALYSIS - RETENTION RISK INTELLIGENCE'
ws3['A1'].font      = make_font(bold=True, size=14)
ws3['A1'].fill      = make_fill(DARK_BLUE)
ws3['A1'].alignment = make_center()
ws3.row_dimensions[1].height = 32

# Driver stats by zone
driver_zone_stats = (
    rides_df[rides_df['status'] == 'completed']
    .groupby(['pickup_zone_id', 'pickup_zone_name', 'pickup_zone_type'])
    .agg(
        total_rides    = ('ride_id',      'count'),
        total_revenue  = ('fare_amount',  'sum'),
        avg_fare       = ('fare_amount',  'mean'),
        avg_wait       = ('wait_time_min','mean'),
        avg_surge      = ('surge_multiplier','mean'),
    )
    .reset_index()
)

driver_counts = (
    drivers_df[drivers_df['status']=='active']
    .groupby('current_zone_id')
    .size()
    .reset_index(name='drivers')
)

driver_zone_stats = driver_zone_stats.merge(
    driver_counts,
    left_on='pickup_zone_id',
    right_on='current_zone_id',
    how='left'
).fillna({'drivers': 0})

driver_zone_stats['est_earnings_per_driver'] = (
    driver_zone_stats['total_revenue']
    / driver_zone_stats['drivers'].replace(0, np.nan)
).round(0)

city_avg = driver_zone_stats['est_earnings_per_driver'].mean()
driver_zone_stats['vs_city_avg_pct'] = (
    (driver_zone_stats['est_earnings_per_driver'] - city_avg)
    / city_avg * 100
).round(1)

driver_zone_stats['equity_class'] = driver_zone_stats[
    'vs_city_avg_pct'
].apply(lambda x:
    'CRITICAL UNDERPAID' if x <= -30
    else 'UNDERPAID'     if x <= -15
    else 'HIGH EARNING'  if x >= 15
    else 'FAIR'
)

driver_zone_stats['retention_risk'] = driver_zone_stats[
    'equity_class'
].map({
    'CRITICAL UNDERPAID': '🔴 CRITICAL',
    'UNDERPAID'         : '🟠 HIGH',
    'FAIR'              : '🟡 MODERATE',
    'HIGH EARNING'      : '🟢 LOW',
})

equity_cols = [
    'Zone ID', 'Zone Name', 'Zone Type',
    'Drivers', 'Total Rides', 'Total Revenue (₹)',
    'Avg Fare (₹)', 'Est. Earnings/Driver (₹)',
    'vs City Avg %', 'Equity Class', 'Retention Risk'
]

driver_display = driver_zone_stats[[
    'pickup_zone_id', 'pickup_zone_name', 'pickup_zone_type',
    'drivers', 'total_rides', 'total_revenue',
    'avg_fare', 'est_earnings_per_driver',
    'vs_city_avg_pct', 'equity_class', 'retention_risk'
]].copy()
driver_display.columns = equity_cols

col_widths3 = [10,28,16,10,12,18,13,22,14,20,16]
for i, w in enumerate(col_widths3, 1):
    ws3.column_dimensions[get_column_letter(i)].width = w

for col_idx, col_name in enumerate(equity_cols, 1):
    col = get_column_letter(col_idx)
    ws3[f'{col}2'] = col_name
    ws3[f'{col}2'].font      = make_font(bold=True, size=10)
    ws3[f'{col}2'].fill      = make_fill(HEADER_BLUE)
    ws3[f'{col}2'].alignment = make_center()
    ws3[f'{col}2'].border    = make_border()
ws3.row_dimensions[2].height = 20

driver_sorted = driver_display.sort_values(
    'vs City Avg %'
).reset_index(drop=True)

equity_colors = {
    'CRITICAL UNDERPAID': 'ffebee',
    'UNDERPAID'         : 'fff3e0',
    'FAIR'              : 'f9fbe7',
    'HIGH EARNING'      : 'e8f5e9',
}

for r_idx, row_data in driver_sorted.iterrows():
    excel_row = r_idx + 3
    cls       = row_data['Equity Class']
    row_color = equity_colors.get(cls, LIGHT_GREY)

    for col_idx, val in enumerate(row_data, 1):
        col = get_column_letter(col_idx)
        if isinstance(val, float):
            val = round(val, 1)
        ws3[f'{col}{excel_row}'] = val
        ws3[f'{col}{excel_row}'].font      = make_font(
            size=9, color=DARK_GREY
        )
        ws3[f'{col}{excel_row}'].fill      = make_fill(row_color)
        ws3[f'{col}{excel_row}'].alignment = make_center()
        ws3[f'{col}{excel_row}'].border    = make_border()
    ws3.row_dimensions[excel_row].height = 16

ws3.freeze_panes = 'A3'
ws3.auto_filter.ref = f"A2:{get_column_letter(len(equity_cols))}2"
print("  ✓ Sheet 3: Driver Earnings Equity")

# ============================================================
# SHEET 4 — DEMAND FORECAST
# ============================================================
ws4 = wb.create_sheet("Demand Forecast")
ws4.sheet_view.showGridLines = False

ws4.merge_cells('A1:F1')
ws4['A1'] = '60-DAY DEMAND FORECAST - GRADIENT BOOSTING MODEL (94.1% ACCURACY)'
ws4['A1'].font      = make_font(bold=True, size=14)
ws4['A1'].fill      = make_fill(DARK_BLUE)
ws4['A1'].alignment = make_center()
ws4.row_dimensions[1].height = 32

# Model metrics summary
metrics = [
    ('Model Algorithm',  'Gradient Boosting Regressor'),
    ('Training Period',  'Jan 2024 - Oct 2024'),
    ('Test Period',      'Oct 2024 - Nov 2024 (30 days)'),
    ('Accuracy (MAPE)',  '94.1% (MAPE: 5.9%)'),
    ('MAE',              '100.6 rides/day'),
    ('RMSE',             '173.6 rides/day'),
    ('Forecast Period',  'Nov 2024 - Jan 2025 (60 days)'),
    ('Peak Predicted',   'Dec 29, 2024 (2,372 rides - Sunburn Festival)'),
]

ws4['A2'] = 'MODEL PERFORMANCE METRICS'
ws4['A2'].font      = make_font(bold=True, size=11)
ws4['A2'].fill      = make_fill(HEADER_BLUE)
ws4['A2'].alignment = make_left()
ws4.merge_cells('A2:F2')
ws4.row_dimensions[2].height = 20

for i, (metric, value) in enumerate(metrics, 3):
    ws4[f'A{i}'] = metric
    ws4[f'A{i}'].font      = make_font(bold=True, size=10, color=DARK_GREY)
    ws4[f'A{i}'].fill      = make_fill('e3f2fd')
    ws4[f'A{i}'].alignment = make_left()
    ws4[f'A{i}'].border    = make_border()
    ws4.merge_cells(f'B{i}:F{i}')
    ws4[f'B{i}'] = value
    ws4[f'B{i}'].font      = make_font(size=10, color=DARK_GREY)
    ws4[f'B{i}'].fill      = make_fill(LIGHT_GREY)
    ws4[f'B{i}'].alignment = make_left()
    ws4[f'B{i}'].border    = make_border()
    ws4.row_dimensions[i].height = 18

# Forecast data
start_row = len(metrics) + 4
ws4[f'A{start_row}'] = 'FORECAST DATA'
ws4[f'A{start_row}'].font      = make_font(bold=True, size=11)
ws4[f'A{start_row}'].fill      = make_fill(HEADER_BLUE)
ws4.merge_cells(f'A{start_row}:F{start_row}')
ws4.row_dimensions[start_row].height = 20

fc_cols = ['Date', 'Predicted Rides',
           'Lower Bound', 'Upper Bound',
           'Day of Week', 'Risk Flag']

col_widths4 = [14,16,14,14,14,16]
for i, w in enumerate(col_widths4, 1):
    ws4.column_dimensions[get_column_letter(i)].width = w

header_row = start_row + 1
for col_idx, col_name in enumerate(fc_cols, 1):
    col = get_column_letter(col_idx)
    ws4[f'{col}{header_row}'] = col_name
    ws4[f'{col}{header_row}'].font      = make_font(bold=True, size=10)
    ws4[f'{col}{header_row}'].fill      = make_fill(DARK_GREY)
    ws4[f'{col}{header_row}'].alignment = make_center()
    ws4[f'{col}{header_row}'].border    = make_border()
ws4.row_dimensions[header_row].height = 18

p90 = forecast_df['predicted_rides'].quantile(0.90)

for r_idx, row_data in forecast_df.iterrows():
    excel_row = r_idx + header_row + 1
    dow       = pd.to_datetime(row_data['date']).strftime('%A')
    is_peak   = row_data['predicted_rides'] >= p90
    risk_flag = '⚡ HIGH DEMAND' if is_peak else '✓ Normal'
    row_color = 'fff3e0' if is_peak else LIGHT_GREY

    values = [
        str(row_data['date'])[:10],
        round(row_data['predicted_rides'], 0),
        round(row_data['lower_bound'], 0),
        round(row_data['upper_bound'], 0),
        dow,
        risk_flag,
    ]
    for col_idx, val in enumerate(values, 1):
        col = get_column_letter(col_idx)
        ws4[f'{col}{excel_row}'] = val
        ws4[f'{col}{excel_row}'].font      = make_font(
            size=9, color=DARK_GREY,
            bold=is_peak
        )
        ws4[f'{col}{excel_row}'].fill      = make_fill(row_color)
        ws4[f'{col}{excel_row}'].alignment = make_center()
        ws4[f'{col}{excel_row}'].border    = make_border()
    ws4.row_dimensions[excel_row].height = 16

print("  ✓ Sheet 4: Demand Forecast")

# ============================================================
# SHEET 5 — EV CHARGING INTELLIGENCE
# ============================================================
ws5 = wb.create_sheet("EV Charging Intelligence")
ws5.sheet_view.showGridLines = False

ws5.merge_cells('A1:J1')
ws5['A1'] = 'EV CHARGING STATION INTELLIGENCE - NETWORK PERFORMANCE & UTILIZATION'
ws5['A1'].font      = make_font(bold=True, size=14)
ws5['A1'].fill      = make_fill(DARK_BLUE)
ws5['A1'].alignment = make_center()
ws5.row_dimensions[1].height = 32

# Network summary
network_summary = (
    stations_df.groupby('network').agg(
        stations        = ('station_id',          'count'),
        operational     = ('is_operational',       'sum'),
        avg_util        = ('avg_utilization',      'mean'),
        total_ports     = ('total_ports',          'sum'),
        monthly_rev     = ('monthly_revenue_inr',  'sum'),
        avg_power_kw    = ('power_kw',             'mean'),
    )
    .reset_index()
    .sort_values('monthly_rev', ascending=False)
)
network_summary['avg_util_pct'] = (
    network_summary['avg_util'] * 100
).round(1)
network_summary['monthly_rev_L'] = (
    network_summary['monthly_rev'] / 100000
).round(1)

net_cols = ['Network', 'Stations', 'Operational',
            'Total Ports', 'Avg Util %',
            'Avg Power kW', 'Monthly Rev (₹L)']

ws5['A2'] = 'NETWORK PERFORMANCE SUMMARY'
ws5.merge_cells('A2:G2')
ws5['A2'].font      = make_font(bold=True, size=11)
ws5['A2'].fill      = make_fill(HEADER_BLUE)
ws5['A2'].alignment = make_left()
ws5.row_dimensions[2].height = 20

net_col_widths = [20,12,14,13,13,15,18]
for i, w in enumerate(net_col_widths, 1):
    ws5.column_dimensions[get_column_letter(i)].width = w

for col_idx, col_name in enumerate(net_cols, 1):
    col = get_column_letter(col_idx)
    ws5[f'{col}3'] = col_name
    ws5[f'{col}3'].font      = make_font(bold=True, size=10)
    ws5[f'{col}3'].fill      = make_fill(DARK_GREY)
    ws5[f'{col}3'].alignment = make_center()
    ws5[f'{col}3'].border    = make_border()
ws5.row_dimensions[3].height = 18

for r_idx, row_data in network_summary.iterrows():
    excel_row = r_idx + 4
    util      = row_data['avg_util_pct']
    row_color = ('e8f5e9' if util >= 60
                 else 'fff3e0' if util >= 45
                 else 'ffebee')

    values = [
        row_data['network'],
        int(row_data['stations']),
        int(row_data['operational']),
        int(row_data['total_ports']),
        f"{util}%",
        round(row_data['avg_power_kw'], 1),
        row_data['monthly_rev_L'],
    ]
    for col_idx, val in enumerate(values, 1):
        col = get_column_letter(col_idx)
        ws5[f'{col}{excel_row}'] = val
        ws5[f'{col}{excel_row}'].font      = make_font(size=10, color=DARK_GREY)
        ws5[f'{col}{excel_row}'].fill      = make_fill(row_color)
        ws5[f'{col}{excel_row}'].alignment = make_center()
        ws5[f'{col}{excel_row}'].border    = make_border()
    ws5.row_dimensions[excel_row].height = 18

# Station detail table
detail_start = len(network_summary) + 6
ws5[f'A{detail_start}'] = 'STATION-LEVEL DETAIL'
ws5.merge_cells(f'A{detail_start}:J{detail_start}')
ws5[f'A{detail_start}'].font      = make_font(bold=True, size=11)
ws5[f'A{detail_start}'].fill      = make_fill(HEADER_BLUE)
ws5[f'A{detail_start}'].alignment = make_left()
ws5.row_dimensions[detail_start].height = 20

station_cols = [
    'Station ID', 'Network', 'Zone Name', 'Zone Type',
    'Charger Type', 'Ports', 'Power kW',
    'Cost/kWh (₹)', 'Utilization %', 'Monthly Rev (₹)'
]

for col_idx, col_name in enumerate(station_cols, 1):
    col = get_column_letter(col_idx)
    ws5[f'{col}{detail_start+1}'] = col_name
    ws5[f'{col}{detail_start+1}'].font      = make_font(bold=True, size=9)
    ws5[f'{col}{detail_start+1}'].fill      = make_fill(DARK_GREY)
    ws5[f'{col}{detail_start+1}'].alignment = make_center()
    ws5[f'{col}{detail_start+1}'].border    = make_border()

stations_sorted = stations_df.sort_values(
    'monthly_revenue_inr', ascending=False
)
for r_idx, row_data in stations_sorted.iterrows():
    excel_row = r_idx + detail_start + 2
    is_op     = row_data['is_operational']
    row_color = LIGHT_GREY if is_op else 'ffebee'
    values = [
        row_data['station_id'],
        row_data['network'],
        row_data['zone_name'],
        row_data['zone_type'],
        row_data['charger_type'].replace('_',' ').title(),
        int(row_data['total_ports']),
        row_data['power_kw'],
        row_data['cost_per_kwh'],
        f"{row_data['avg_utilization']*100:.0f}%",
        int(row_data['monthly_revenue_inr']),
    ]
    for col_idx, val in enumerate(values, 1):
        col = get_column_letter(col_idx)
        ws5[f'{col}{excel_row}'] = val
        ws5[f'{col}{excel_row}'].font      = make_font(size=9, color=DARK_GREY)
        ws5[f'{col}{excel_row}'].fill      = make_fill(row_color)
        ws5[f'{col}{excel_row}'].alignment = make_center()
        ws5[f'{col}{excel_row}'].border    = make_border()
    ws5.row_dimensions[excel_row].height = 15

print("  ✓ Sheet 5: EV Charging Intelligence")

# ============================================================
# SHEET 6 — ML MODEL RESULTS
# ============================================================
ws6 = wb.create_sheet("ML Model Results")
ws6.sheet_view.showGridLines = False

ws6.merge_cells('A1:F1')
ws6['A1'] = 'XGBOOST DEMAND SPIKE PREDICTION - MODEL PERFORMANCE REPORT'
ws6['A1'].font      = make_font(bold=True, size=14)
ws6['A1'].fill      = make_fill(DARK_BLUE)
ws6['A1'].alignment = make_center()
ws6.row_dimensions[1].height = 32

col_widths6 = [28, 20, 20, 20, 20, 20]
for i, w in enumerate(col_widths6, 1):
    ws6.column_dimensions[get_column_letter(i)].width = w

ml_sections = [
    ('MODEL CONFIGURATION', [
        ('Algorithm',            'XGBoost Classifier'),
        ('n_estimators',         '500'),
        ('Best Iteration',       '176'),
        ('Max Depth',            '6'),
        ('Learning Rate',        '0.05'),
        ('Subsample',            '0.8'),
        ('Colsample by Tree',    '0.8'),
        ('Scale Pos Weight',     f"{(y_train==0).sum()/(y_train==1).sum():.3f}"),
    ]),
    ('DATASET STATISTICS', [
        ('Total Zone-Hour Records', f"{len(zone_hour):,}"),
        ('Features Engineered',    '19'),
        ('Train Size',             f"{len(X_train):,}"),
        ('Test Size',              f"{len(X_test):,}"),
        ('Spike Rate (Train)',      f"{y_train.mean()*100:.1f}%"),
        ('Spike Rate (Test)',       f"{y_test.mean()*100:.1f}%"),
    ]),
    ('MODEL PERFORMANCE', [
        ('Accuracy',         '90.53%'),
        ('ROC-AUC Score',    '0.9627'),
        ('Precision',        '1.0000  ← ZERO false alarms'),
        ('Recall',           '0.8474'),
        ('F1 Score',         '0.9174'),
        ('True Positives',   f"{tp:,}"),
        ('False Positives',  f"{fp}  ← Zero false alarms"),
        ('False Negatives',  f"{fn:,}"),
        ('True Negatives',   f"{tn:,}"),
    ]),
    ('SHAP FEATURE IMPORTANCE', [
        ('1. Zone Demand Weight',  '3.4084 - Primary driver'),
        ('2. Avg Wait Time',       '0.6165 - Service stress signal'),
        ('3. Zone Type',           '0.5644 - Tech parks spike most'),
        ('4. Completion Rate',     '0.5600 - Demand capacity proxy'),
        ('5. Evening Peak Hour',   '0.0701 - 6pm surge effect'),
        ('6. Unmet Demand Rate',   '0.0577 - Supply shortage signal'),
        ('7. Hour of Day',         '0.0477 - Time pattern'),
        ('8. Morning Peak Hour',   '0.0289 - 8am surge effect'),
    ]),
    ('SCENARIO PREDICTION', [
        ('Scenario',          'Koramangala 1B | Friday 6pm | Heavy Rain'),
        ('Spike Probability', '99.6%'),
        ('Prediction',        '⚡ SPIKE PREDICTED'),
        ('Top Driver',        'Completion Rate ↑ spike by 4.07'),
        ('2nd Driver',        'Avg Wait Time ↑ spike by 1.94'),
        ('3rd Driver',        'Zone Demand Weight ↓ spike by 0.72'),
        ('Business Action',   'Pre-position 8 additional drivers by 5:30pm'),
    ]),
]

current_row = 2
for section_title, section_data in ml_sections:
    ws6.merge_cells(f'A{current_row}:F{current_row}')
    ws6[f'A{current_row}'] = section_title
    ws6[f'A{current_row}'].font      = make_font(bold=True, size=11)
    ws6[f'A{current_row}'].fill      = make_fill(HEADER_BLUE)
    ws6[f'A{current_row}'].alignment = make_left()
    ws6.row_dimensions[current_row].height = 22
    current_row += 1

    for metric, value in section_data:
        ws6[f'A{current_row}'] = metric
        ws6[f'A{current_row}'].font      = make_font(bold=True, size=10, color=DARK_GREY)
        ws6[f'A{current_row}'].fill      = make_fill('e3f2fd')
        ws6[f'A{current_row}'].alignment = make_left()
        ws6[f'A{current_row}'].border    = make_border()

        ws6.merge_cells(f'B{current_row}:F{current_row}')
        ws6[f'B{current_row}'] = value
        ws6[f'B{current_row}'].font      = make_font(size=10, color=DARK_GREY)
        ws6[f'B{current_row}'].fill      = make_fill(LIGHT_GREY)
        ws6[f'B{current_row}'].alignment = make_left()
        ws6[f'B{current_row}'].border    = make_border()
        ws6.row_dimensions[current_row].height = 18
        current_row += 1

    current_row += 1

print("  ✓ Sheet 6: ML Model Results")

# ============================================================
# SAVE WORKBOOK
# ============================================================
excel_path = os.path.join(
    EXCEL_OUTPUT_PATH,
    "Urban_Mobility_Intelligence_Report.xlsx"
)
wb.save(excel_path)

print(f"\n{'='*60}")
print(f"  EXCEL WORKBOOK SAVED SUCCESSFULLY")
print(f"{'='*60}")
print(f"  File     : Urban_Mobility_Intelligence_Report.xlsx")
print(f"  Location : outputs/excel/")
print(f"  Sheets   : 6")
print(f"  Sheet 1  : Executive Summary")
print(f"  Sheet 2  : Zone Intelligence (200 zones)")
print(f"  Sheet 3  : Driver Earnings Equity")
print(f"  Sheet 4  : Demand Forecast (60 days)")
print(f"  Sheet 5  : EV Charging Intelligence")
print(f"  Sheet 6  : ML Model Results")
print(f"\n✓ NOTEBOOK 06 COMPLETE - Excel report generated.")
print(f"  Next → Phase 9 - Power BI Dashboard")

  ✓ Sheet 1: Executive Summary
  ✓ Sheet 2: Zone Intelligence
  ✓ Sheet 3: Driver Earnings Equity
  ✓ Sheet 4: Demand Forecast
  ✓ Sheet 5: EV Charging Intelligence


NameError: name 'y_train' is not defined

In [6]:
# ── Fix: hardcode ML variables not available in this notebook
y_train_spike_rate = 62.1  # from notebook 05
y_test_spike_rate  = 62.1
zone_hour_count    = 84941
x_train_size       = 67952
x_test_size        = 16989
tp = 8938
fp = 0
fn = 1609
tn = 6442

ml_sections = [
    ('MODEL CONFIGURATION', [
        ('Algorithm',            'XGBoost Classifier'),
        ('n_estimators',         '500'),
        ('Best Iteration',       '176'),
        ('Max Depth',            '6'),
        ('Learning Rate',        '0.05'),
        ('Subsample',            '0.8'),
        ('Colsample by Tree',    '0.8'),
        ('Scale Pos Weight',     '0.610'),
    ]),
    ('DATASET STATISTICS', [
        ('Total Zone-Hour Records', f"{zone_hour_count:,}"),
        ('Features Engineered',    '19'),
        ('Train Size',             f"{x_train_size:,}"),
        ('Test Size',              f"{x_test_size:,}"),
        ('Spike Rate (Train)',      f"{y_train_spike_rate:.1f}%"),
        ('Spike Rate (Test)',       f"{y_test_spike_rate:.1f}%"),
    ]),
    ('MODEL PERFORMANCE', [
        ('Accuracy',         '90.53%'),
        ('ROC-AUC Score',    '0.9627'),
        ('Precision',        '1.0000  \u2190 ZERO false alarms'),
        ('Recall',           '0.8474'),
        ('F1 Score',         '0.9174'),
        ('True Positives',   f"{tp:,}"),
        ('False Positives',  f"{fp}  \u2190 Zero false alarms"),
        ('False Negatives',  f"{fn:,}"),
        ('True Negatives',   f"{tn:,}"),
    ]),
    ('SHAP FEATURE IMPORTANCE', [
        ('1. Zone Demand Weight',  '3.4084 - Primary driver'),
        ('2. Avg Wait Time',       '0.6165 - Service stress signal'),
        ('3. Zone Type',           '0.5644 - Tech parks spike most'),
        ('4. Completion Rate',     '0.5600 - Demand capacity proxy'),
        ('5. Evening Peak Hour',   '0.0701 - 6pm surge effect'),
        ('6. Unmet Demand Rate',   '0.0577 - Supply shortage signal'),
        ('7. Hour of Day',         '0.0477 - Time pattern'),
        ('8. Morning Peak Hour',   '0.0289 - 8am surge effect'),
    ]),
    ('SCENARIO PREDICTION', [
        ('Scenario',          'Koramangala 1B | Friday 6pm | Heavy Rain'),
        ('Spike Probability', '99.6%'),
        ('Prediction',        'SPIKE PREDICTED'),
        ('Top Driver',        'Completion Rate increases spike by 4.07'),
        ('2nd Driver',        'Avg Wait Time increases spike by 1.94'),
        ('3rd Driver',        'Zone Demand Weight decreases spike by 0.72'),
        ('Business Action',   'Pre-position 8 additional drivers by 5:30pm'),
    ]),
]

current_row = 2
for section_title, section_data in ml_sections:
    ws6.merge_cells(f'A{current_row}:F{current_row}')
    ws6[f'A{current_row}'] = section_title
    ws6[f'A{current_row}'].font      = make_font(bold=True, size=11)
    ws6[f'A{current_row}'].fill      = make_fill(HEADER_BLUE)
    ws6[f'A{current_row}'].alignment = make_left()
    ws6.row_dimensions[current_row].height = 22
    current_row += 1

    for metric, value in section_data:
        ws6[f'A{current_row}'] = metric
        ws6[f'A{current_row}'].font      = make_font(bold=True, size=10, color=DARK_GREY)
        ws6[f'A{current_row}'].fill      = make_fill('e3f2fd')
        ws6[f'A{current_row}'].alignment = make_left()
        ws6[f'A{current_row}'].border    = make_border()

        ws6.merge_cells(f'B{current_row}:F{current_row}')
        ws6[f'B{current_row}'] = value
        ws6[f'B{current_row}'].font      = make_font(size=10, color=DARK_GREY)
        ws6[f'B{current_row}'].fill      = make_fill(LIGHT_GREY)
        ws6[f'B{current_row}'].alignment = make_left()
        ws6[f'B{current_row}'].border    = make_border()
        ws6.row_dimensions[current_row].height = 18
        current_row += 1

    current_row += 1

print("  ✓ Sheet 6: ML Model Results")

# ── Save workbook ─────────────────────────────────────────────
excel_path = os.path.join(
    EXCEL_OUTPUT_PATH,
    "Urban_Mobility_Intelligence_Report.xlsx"
)
wb.save(excel_path)

print(f"\n{'='*60}")
print(f"  EXCEL WORKBOOK SAVED SUCCESSFULLY")
print(f"{'='*60}")
print(f"  File   : Urban_Mobility_Intelligence_Report.xlsx")
print(f"  Sheets : 6")
print(f"  Sheet 1: Executive Summary")
print(f"  Sheet 2: Zone Intelligence (200 zones)")
print(f"  Sheet 3: Driver Earnings Equity")
print(f"  Sheet 4: Demand Forecast (60 days)")
print(f"  Sheet 5: EV Charging Intelligence")
print(f"  Sheet 6: ML Model Results")
print(f"\n✓ NOTEBOOK 06 COMPLETE - Excel report generated.")
print(f"  Next -> Phase 9 - Power BI Dashboard")

  ✓ Sheet 6: ML Model Results

  EXCEL WORKBOOK SAVED SUCCESSFULLY
  File   : Urban_Mobility_Intelligence_Report.xlsx
  Sheets : 6
  Sheet 1: Executive Summary
  Sheet 2: Zone Intelligence (200 zones)
  Sheet 3: Driver Earnings Equity
  Sheet 4: Demand Forecast (60 days)
  Sheet 5: EV Charging Intelligence
  Sheet 6: ML Model Results

✓ NOTEBOOK 06 COMPLETE - Excel report generated.
  Next -> Phase 9 - Power BI Dashboard
